[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/tracing/dspy_tracing_tutorial.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">追踪与评估 DSPy 应用</h1>

DSPy 是自动优化提示词和微调语言模型的框架，提供：

- 可组合的声明式 API，以“模块”形式描述 LLM 应用架构，灵感源于 PyTorch 的 `nn.Module`
- 名为“teleprompter”的编译器，针对特定任务优化用户定义的模块。“teleprompter”意指“远程提示”，可以涉及选择少样本示例、生成提示词或微调语言模型

Phoenix 将编译后的 DSPy 模块每次调用的底层结构可视化，并根据延迟、token 数及其他评估指标展示存在问题的 span，让应用具备*可观测性*。

本教程将：
- 构建并编译 DSPy 模块，通过检索增强生成回答 [HotpotQA 数据集](https://hotpotqa.github.io/wiki-readme.html)中的问题
- 使用记录 LLM 遥测数据的开放标准 [OpenInference](https://github.com/Arize-ai/openinference) 为应用添加埋点
- 检查应用的追踪和 span，理解 DSPy 前向传播的内部过程

ℹ️ 本 Notebook 需要 OpenAI API 密钥。


## 1. 安装依赖并导入库

安装 Phoenix、DSPy 及其他依赖。

In [ ]:
!pip install arize-phoenix "dspy>=2.5.0" "openinference-instrumentation-dspy>=0.1.13" openinference-instrumentation-litellm opentelemetry-exporter-otlp

导入库。

In [ ]:
import os
from getpass import getpass

import dspy
import openai

import phoenix as px

## 2. 配置 OpenAI API 密钥

如果尚未通过环境变量设置 OpenAI API 密钥，请在此设置。

In [ ]:
if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥： ")
openai.api_key = openai_api_key
os.environ["OPENAI_API_KEY"] = openai_api_key

## 3. 配置模块组件

模块由语言模型（本例为 OpenAI GPT-4）和检索器（本例为 ColBERTv2）等组件组成，类似于 PyTorch 模块中的各层。

In [ ]:
lm = dspy.LM("openai/gpt-4", cache=False)
colbertv2_wiki17_abstracts = dspy.ColBERTv2(
    url="http://20.102.90.50:2017/wiki17_abstracts"  # 托管 ColBERTv2 服务的端点
)

dspy.settings.configure(lm=lm, rm=colbertv2_wiki17_abstracts)

## 4. 加载数据

加载 HotpotQA 数据集的一个子集。

In [ ]:
from dspy.datasets import HotPotQA

# 加载数据集。
dataset = HotPotQA(train_seed=1, train_size=20, eval_seed=2023, dev_size=50, test_size=10)

# 告诉 DSPy，'question' 字段是输入，其他字段是标签或元数据。
trainset = [x.with_inputs("question") for x in dataset.train]
devset = [x.with_inputs("question") for x in dataset.dev]

print(f"训练集大小： {len(trainset)}")
print(f"开发集大小： {len(devset)}")

训练集中的每个样本都包含问题和人工标注的答案。

In [ ]:
train_example = trainset[0]
train_example

开发集样本还包含第三个字段，记录相关维基百科文章的标题。

In [ ]:
dev_example = devset[18]
dev_example

## 5. 定义 RAG 模块

定义接收 `context` 和 `question` 两个输入、输出 `answer` 的签名。签名提供：

- 语言模型要解决的子任务描述
- 语言模型输入字段的描述
- 语言模型必须生成的输出字段描述

In [ ]:
class GenerateAnswer(dspy.Signature):
    """使用简短事实回答问题。"""

    context = dspy.InputField(desc="可能包含相关事实")
    question = dspy.InputField()
    answer = dspy.OutputField(desc="通常为 1–5 个词")

通过继承 `dspy.Module` 并重写 `forward` 方法来定义模块。

In [ ]:
class RAG(dspy.Module):
    def __init__(self, num_passages=3):
        super().__init__()
        self.retrieve = dspy.Retrieve(k=num_passages)
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question):
        context = self.retrieve(question).passages
        prediction = self.generate_answer(context=context, question=question)
        return dspy.Prediction(context=context, answer=prediction.answer)

此模块结合检索增强生成（使用前面配置的 ColBERTv2 检索器）和思维链，生成面向用户的最终回答。

## 6. 编译 RAG 模块

本例使用默认的 `BootstrapFewShot` teleprompter，从训练集中选择优质示例加入最终提示词。

In [ ]:
from dspy.teleprompt import BootstrapFewShot


# 验证逻辑：检查预测答案是否正确。
# 同时检查检索上下文是否确实包含答案。
def validate_context_and_answer(example, pred, trace=None):
    answer_EM = dspy.evaluate.answer_exact_match(example, pred)
    answer_PM = dspy.evaluate.answer_passage_match(example, pred)
    return answer_EM and answer_PM


input_module = RAG()
teleprompter = BootstrapFewShot(metric=validate_context_and_answer)
compiled_module = teleprompter.compile(input_module, trainset=trainset)

## 7. 为 DSPy 添加埋点并启动 Phoenix

RAG 程序已经编译完成，现在查看其内部执行过程。

启动 Phoenix，在后台收集已添加埋点的 DSPy 应用中的 span 和追踪。

In [ ]:
phoenix_session = px.launch_app()

接下来使用 [OpenInference](https://github.com/Arize-ai/openinference/tree/main/spec) 为应用添加埋点。它是构建于 [OpenTelemetry](https://opentelemetry.io/) 之上的开放标准，用于捕获和存储 LLM 应用执行。OpenInference 提供遥测数据，帮助理解 LLM 调用及其周边上下文，包括向量库检索、外部工具或 API 的使用等。

DSPy 底层使用 LiteLLM 调用 LLM，因此在此添加 `LiteLLMInstrumentor` 以获取 LLM span 的 token 数。

In [ ]:
from openinference.instrumentation.dspy import DSPyInstrumentor
from openinference.instrumentation.litellm import LiteLLMInstrumentor
from phoenix.otel import register

register(endpoint="http://127.0.0.1:6006/v1/traces")
DSPyInstrumentor().instrument(skip_dep_check=True)
LiteLLMInstrumentor().instrument(skip_dep_check=True)

## 8. 运行应用

在开发集上运行 DSPy 应用。

In [ ]:
for example in devset:
    question = example["question"]
    prediction = compiled_module(question)
    print("问题")
    print("========")
    print(question)
    print()
    print("预测答案")
    print("================")
    print(prediction.answer)
    print()
    print("检索上下文（已截断）")
    print(f"{[c[:200] + '...' for c in prediction.context]}")
    print()
    print()

在 Phoenix 界面中检查 DSPy 模块的架构。

In [ ]:
print(phoenix_session.url)

注意以下几点：

- 每条追踪中的 span 对应自定义 `dspy.Module` 子类的 `forward` 方法中的各个步骤
- `ColBERTv2` 调用显示为检索器 span，展示每次前向传播检索到的文档和分数
- LLM span 包含完整格式化的提示词，其中包括 DSPy 在编译阶段计算得到的少样本示例

![DSPy 追踪和 span 导览，重点展示检索器与 LLM span](https://storage.googleapis.com/arize-phoenix-assets/assets/docs/notebooks/dspy-tracing-tutorial/dspy_spans_and_traces.gif)

你已经使用 DSPy 构建了包含困难负例段落和思维链的多示例提示词，并通过 Phoenix 观察 DSPy 的内部执行过程，理解前向传播的工作方式。